# Extension 2: MLLMU-Bench Generalization Study

**Objective**: Evaluate whether unlearning method rankings generalize across benchmarks (FIUBench vs MLLMU-Bench).

**Proposal Requirements**:
- Model: LLaVA-Phi-3-mini-3B with LoRA (fine-tuning + unlearning)
- Dataset: MLLMU-Bench (100 fictional identities: 50 forget, 50 retain)
- Methods: GA, GD, KL, PO (FIUBench suite, NOT NPO)
- Metrics: D_unlearn, D_retain, MMBench
- Primary Analysis: Kendall's τ correlation between FIUBench and MLLMU-Bench method rankings
- Secondary: CMLD (cross-modal leakage) comparison, radar charts

**Key Constraint**: Cannot reproduce PULSE's absolute numbers—only test if relative method rankings are consistent across benchmarks.

## Cell 1: Setup & Paths

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
import subprocess

result = subprocess.run(
    "git clone https://YOUR_TOKEN@github.com/akashyall34/FIUBench_Reproducing.git /content/FIUBench_Reproducing",
    shell=True, capture_output=True, text=True
)
print(result.stdout or result.stderr)

fatal: destination path '/content/FIUBench_Reproducing' already exists and is not an empty directory.



In [22]:
import subprocess
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'

print(f"Pulling latest changes from GitHub...\n")

# Stash local changes first to avoid conflicts
print("Stashing local changes...")
result = subprocess.run(
    "git stash",
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    shell=True
)
if result.returncode == 0 and result.stdout.strip():
    print(result.stdout)
else:
    print("(no local changes to stash)")

# Now pull
result = subprocess.run(
    "git pull",
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    shell=True
)

if result.returncode == 0:
    print("✅ Repository updated")
    print(result.stdout)
else:
    print("❌ Pull failed")
    print(result.stderr)

Pulling latest changes from GitHub...

Stashing local changes...
No local changes to save

✅ Repository updated
Already up to date.



In [ ]:
import subprocess, sys

deps = [
    "torch==2.4.1",
    "transformers==4.48.0",
    "xtuner==0.2.0",
    "accelerate==0.34.2",
    "datasets==2.21.0",
    "peft==0.13.2",
    "pillow",
    "scikit-learn",
    "rouge-score",
    "open-clip-torch",
    "hf_transfer",
]

for dep in deps:
    subprocess.run(f"{sys.executable} -m pip install -q {dep}", shell=True)

import transformers, torch
print(f"✅ torch={torch.__version__}  transformers={transformers.__version__}")

In [24]:
import os
import sys
import torch
import json
import subprocess
import time
from pathlib import Path
from huggingface_hub import snapshot_download

# ─── Detect Colab Environment ──────────────────────────────────────────────
try:
    from google.colab import drive
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

# ─── Mount Google Drive (Colab only) ───────────────────────────────────────
if IN_COLAB:
    drive.mount('/content/drive', force_remount=False)
    print("Google Drive mounted at /content/drive")

# ─── Device ───────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')

# ─── Configuration (Per Proposal) ──────────────────────────────────────────
BASE_MODEL = 'xtuner/llava-phi-3-mini-hf'  # Proposal requirement
FORGET_SPLIT = 'forget_5'  # Pre-made split in MLLMU-Bench
RETAIN_SPLIT = 'retain_95'  # Corresponding retain set

# ─── Paths (Colab vs Local) ───────────────────────────────────────────────
if IN_COLAB:
    # Google Drive paths - use fiubench_checkpoints
    CKPT_DIR = Path('/content/drive/MyDrive/fiubench_checkpoints/extension2')
    OUTPUT_DIR = Path('/content/drive/MyDrive/fiubench_checkpoints/extension2/outputs')
    CACHE_DIR = Path('/content/drive/MyDrive/fiubench_cache/mllmu')
    MLLMU_DIR = Path('/content/FIUBench_Reproducing/MLLMU-Bench')  # From cloned repo
    print("Using Google Drive paths (fiubench_checkpoints)")
else:
    # Local paths
    WORK_DIR = Path('/Users/akashy/Library/CloudStorage/OneDrive-UniversityofSouthFlorida/Projects/FIUBench_Reproducing')
    CKPT_DIR = WORK_DIR / 'checkpoints' / 'extension2'
    OUTPUT_DIR = WORK_DIR / 'outputs' / 'extension2'
    CACHE_DIR = WORK_DIR / 'cache' / 'mllmu'
    MLLMU_DIR = WORK_DIR / 'MLLMU-Bench'
    print("Using local paths")

DATA_DIR = CACHE_DIR / 'MLLMU-Bench'

# Create directories
for d in [CKPT_DIR, OUTPUT_DIR, CACHE_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'\nModel: {BASE_MODEL}')
print(f'Split: {FORGET_SPLIT} / {RETAIN_SPLIT}')
print(f'Paths:')
print(f'  MLLMU-Bench repo: {MLLMU_DIR}')
print(f'  Data cache: {DATA_DIR}')
print(f'  Checkpoints: {CKPT_DIR}')
print(f'  Outputs: {OUTPUT_DIR}')

# ─── HuggingFace Authentication ────────────────────────────────────────────
if 'HF_TOKEN' not in os.environ:
    print('\nHF_TOKEN set (via login() in Cell 0)')

# ─── Download config ──────────────────────────────────────────────────────
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('\nSetup complete')

Running in Google Colab
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted at /content/drive
Device: cuda
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1GB
Using Google Drive paths (fiubench_checkpoints)

Model: xtuner/llava-phi-3-mini-hf
Split: forget_5 / retain_95
Paths:
  MLLMU-Bench repo: /content/FIUBench_Reproducing/MLLMU-Bench
  Data cache: /content/drive/MyDrive/fiubench_cache/mllmu/MLLMU-Bench
  Checkpoints: /content/drive/MyDrive/fiubench_checkpoints/extension2
  Outputs: /content/drive/MyDrive/fiubench_checkpoints/extension2/outputs

HF_TOKEN set (via login() in Cell 0)

Setup complete


## Cell 3: Fine-tune Vanilla Model on MLLMU-Bench

In [25]:
print('='*60)
print('Downloading MLLMU-Bench Dataset & Pre-Made Splits')
print('='*60)

# ─── Dataset ───────────────────────────────────────────────────────────────
try:
    snapshot_download(
        repo_id='MLLMMU/MLLMU-Bench',
        local_dir=str(DATA_DIR),
        repo_type='dataset',
    )
    print(f'✅ Dataset downloaded to {DATA_DIR}')
except Exception as e:
    print(f'⚠️  Dataset error (may already exist): {e}')

# Verify pre-made splits exist
print('\n' + '-'*60)
print('Verification of Pre-Made Splits:')
print('-'*60)

forget_path = DATA_DIR / FORGET_SPLIT
retain_path = DATA_DIR / RETAIN_SPLIT

forget_files = list(forget_path.glob('*.json')) + list(forget_path.glob('*.parquet'))
retain_files = list(retain_path.glob('*.json')) + list(retain_path.glob('*.parquet'))

if forget_files:
    print(f'✅ {FORGET_SPLIT}: {len(forget_files)} files found')
else:
    print(f'⚠️  {FORGET_SPLIT}: NO FILES FOUND')
    print(f'   Expected path: {forget_path}')

if retain_files:
    print(f'✅ {RETAIN_SPLIT}: {len(retain_files)} files found')
else:
    print(f'⚠️  {RETAIN_SPLIT}: NO FILES FOUND')
    print(f'   Expected path: {retain_path}')

# Also verify fine-tuning data
ft_data = list((DATA_DIR / 'ft_Data').glob('*.parquet')) if (DATA_DIR / 'ft_Data').exists() else []
test_set = list((DATA_DIR / 'Test_Set').glob('*.parquet')) if (DATA_DIR / 'Test_Set').exists() else []

if ft_data:
    print(f'✅ ft_Data: {len(ft_data)} parquet files')
else:
    print(f'⚠️  ft_Data: NOT FOUND')

if test_set:
    print(f'✅ Test_Set: {len(test_set)} parquet files')
else:
    print(f'⚠️  Test_Set: NOT FOUND')

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

✅ Dataset downloaded to /content/drive/MyDrive/fiubench_cache/mllmu/MLLMU-Bench

------------------------------------------------------------
Verification of Pre-Made Splits:
------------------------------------------------------------
✅ forget_5: 1 files found
✅ retain_95: 1 files found
✅ ft_Data: 1 parquet files
✅ Test_Set: 2 parquet files


## Cell 3: Fine-tune Vanilla Model on MLLMU-Bench

In [28]:
import subprocess

# Apply fix to finetune.py in Colab environment
finetune_path = '/content/FIUBench_Reproducing/MLLMU-Bench/finetune.py'
with open(finetune_path, 'r') as f:
    content = f.read()

# Replace the strict startswith check with flexible "llava" in check
content = content.replace(
    'if model_id.startswith("llava"):',
    'if "llava" in model_id.lower():'
)

with open(finetune_path, 'w') as f:
    f.write(content)

print("✅ finetune.py patched for xtuner/llava models")


✅ finetune.py patched for xtuner/llava models


In [ ]:
from huggingface_hub import snapshot_download
import os

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

MODEL_CACHE = CACHE_DIR / 'xtuner_llava_phi'
MODEL_CACHE.mkdir(parents=True, exist_ok=True)

print("Downloading model...")
try:
    snapshot_download(
        repo_id="xtuner/llava-phi-3-mini-hf",
        local_dir=str(MODEL_CACHE),
        ignore_patterns=["*.msgpack", "*.h5", "flax_model*"],
    )
    print(f"✅ Model cached to {MODEL_CACHE}")
except Exception as e:
    print(f"❌ Download failed: {e}")


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

In [29]:
print('\n' + '='*60)
print('Fine-tuning Vanilla Model on MLLMU-Bench')
print('='*60)
print('📝 Per proposal: fine-tune LLaVA-Phi-3-mini with LoRA')

vanilla_local = CKPT_DIR / 'vanilla_model'
vanilla_local.mkdir(parents=True, exist_ok=True)

ft_data_path = list((DATA_DIR).glob('ft_Data/*.parquet'))[0] if list((DATA_DIR).glob('ft_Data/*.parquet')) else None

if not ft_data_path:
    print('⚠️  ERROR: MLLMU-Bench fine-tuning data not found')
    print(f'   Expected: {DATA_DIR}/ft_Data/*.parquet')
    print(f'   Download Cell 2 first')
else:
    cmd = (
        f"cd {MLLMU_DIR} && "
        f"python finetune.py "
        f"--model_id {BASE_MODEL} "
        f"--save_dir {vanilla_local} "
        f"--data_dir {ft_data_path} "
        f"--batch_size 4 "
        f"--lr 2e-5 "
        f"--num_epochs 5 "
        f"--max_length 384 "
        f"2>&1 | tee /tmp/vanilla_finetune.log"
    )

    print(f'\nCommand:')
    print(f'  cd {MLLMU_DIR}')
    print(f'  python finetune.py --model_id {BASE_MODEL} --batch_size 4 --lr 2e-5 --epochs 5')
    print(f'\nStarting fine-tuning')

    t0 = time.time()
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)

    for line in proc.stdout:
        print(line, end='', flush=True)

    proc.wait()
    ret = proc.returncode

    elapsed = time.time() - t0
    h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
    print(f'\nExit: {ret}  |  Time: {h}h {m}m {s}s')

    if ret == 0:
        print(f'✅ Vanilla model fine-tuned and saved to {vanilla_local}')
        print(f'   LoRA adapters merged into model weights')
        files = list(vanilla_local.glob("*"))
        print(f'   Files: {len(files)} total')
    else:
        print(f'❌ Fine-tuning failed (exit {ret})')
        print(f'   Check /tmp/vanilla_finetune.log')


Fine-tuning Vanilla Model on MLLMU-Bench
📝 Per proposal: fine-tune LLaVA-Phi-3-mini with LoRA

Command:
  cd /content/FIUBench_Reproducing/MLLMU-Bench
  python finetune.py --model_id xtuner/llava-phi-3-mini-hf --batch_size 4 --lr 2e-5 --epochs 5

Starting fine-tuning
2026-04-26 19:56:59.492756: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-26 19:56:59.511305: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777233419.533961    9070 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777233419.541627    9070 cuda_blas.cc:1

## Cell 4: Train GA/GD/KL/PO Unlearning Methods

In [ ]:
print('\n' + '='*60)
print('Train FIUBench-Equivalent Methods on MLLMU-Bench')
print('='*60)
print('📝 Using MLLMU-Bench baseline scripts (FIUBench-equivalent methods)')
print('   GA, GD (GA_Difference), KL (KL_Min)')

# Methods available in MLLMU-Bench baselines
# GA.py = Gradient Ascent
# GA_Difference.py = Gradient Difference (FIUBench's GD)
# KL_Min.py = KL Minimization
METHODS = {
    'GA': 'GA.py',
    'GD': 'GA_Difference.py',
    'KL': 'KL_Min.py',
}

vanilla_local = CKPT_DIR / 'vanilla_model'
forget_data_path = DATA_DIR / FORGET_SPLIT
retain_data_path = DATA_DIR / RETAIN_SPLIT

print(f'\n⚠️  NOTE: MLLMU-Bench baselines implement GA, GD, KL')
print(f'   PO (idk/preference optimization) not available in MLLMU baselines')
print(f'   Using 3 methods per MLLMU-Bench baseline suite')
print(f'\nData paths:')
print(f'  Forget set: {forget_data_path}')
print(f'  Retain set: {retain_data_path}\n')

for method_name, script_name in METHODS.items():
    method_local = CKPT_DIR / f'{method_name}'
    method_local.mkdir(parents=True, exist_ok=True)

    print(f'\n--- {method_name} ({script_name}) ---')
    print(f'  Base model: {vanilla_local}')
    print(f'  Output: {method_local}')
    
    # MLLMU-Bench baseline command using pre-made splits
    cmd = (
        f"cd {MLLMU_DIR}/baselines && "
        f"python {script_name} "
        f"--model_id {BASE_MODEL} "
        f"--vanilla_dir {vanilla_local} "
        f"--data_path {forget_data_path} "
        f"--retain_path {retain_data_path} "
        f"--save_dir {method_local} "
        f"--batch_size 4 "
        f"--lr 2e-5 "
        f"--num_epochs 1 "
        f"--max_length 384 "
        f"2>&1 | tee /tmp/{method_name.lower()}_unlearning.log"
    )
    
    print(f'\n  Starting {method_name} unlearning...')
    t0 = time.time()
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    
    for line in proc.stdout:
        print(line, end='', flush=True)
    
    proc.wait()
    ret = proc.returncode
    
    elapsed = time.time() - t0
    h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
    
    if ret == 0:
        print(f'\n  ✅ {method_name} saved | Time: {h}h {m}m {s}s')
    else:
        print(f'\n  ❌ {method_name} failed (exit {ret})')
        print(f'     Check /tmp/{method_name.lower()}_unlearning.log')

print(f'\n✅ Unlearning training complete (3 FIUBench-equivalent methods)')

## Cell 5: Evaluate All Checkpoints with PULSE Metrics

In [ ]:
import pandas as pd
import numpy as np

print('\n' + '='*60)
print('Evaluation: MLLMU-Bench 3-Axis Metrics')
print('='*60)

# Models to evaluate
models_to_eval = {
    'vanilla': CKPT_DIR / 'vanilla_model',
    'GA': CKPT_DIR / 'GA',
    'GD': CKPT_DIR / 'GD',
    'KL': CKPT_DIR / 'KL',
}

print('\nModels to evaluate:')
for name, path in models_to_eval.items():
    exists = '✅' if path.exists() else '⏳'
    print(f'  {exists} {name:8s}: {path}')

# ─── Run eval.py for each model ────────────────────────────────────────────
print('\n' + '-'*60)
print('Running eval.py (3 PULSE metrics: D_unlearn, D_retain, MMBench)')
print('-'*60)

results_by_method = {}

for model_name, model_path in models_to_eval.items():
    if not model_path.exists():
        print(f'\n⏳ Skipping {model_name} (checkpoint not ready)')
        continue
    
    print(f'\n--- Evaluating {model_name} ---')
    
    cmd = (
        f"cd {MLLMU_DIR} && "
        f"python eval.py "
        f"--model_id {BASE_MODEL} "
        f"--cache_path {model_path} "
        f"--test_data {DATA_DIR}/Test_Set "
        f"--few_shot_data {DATA_DIR}/Full_Set/train-00000-of-00001.parquet "
        f"--data_split_folder {DATA_DIR} "
        f"--celebrity_data {DATA_DIR}/Retain_Set/train-00000-of-00001.parquet "
        f"--output_file {model_name}_results "
        f"--output_folder {OUTPUT_DIR} "
        f"2>&1 | tee {OUTPUT_DIR}/{model_name}_eval.log"
    )
    
    t0 = time.time()
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    
    for line in proc.stdout:
        print(line, end='', flush=True)
    
    proc.wait()
    ret = proc.returncode
    
    elapsed = time.time() - t0
    h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
    
    if ret == 0:
        print(f'\n✅ {model_name} evaluation complete | Time: {h}h {m}m {s}s')
        results_by_method[model_name] = f'{model_name}_results.json'
    else:
        print(f'\n❌ {model_name} evaluation failed (exit {ret})')

# ─── Load and display MLLMU-Bench results ──────────────────────────────────
print('\n' + '='*60)
print('MLLMU-Bench Results (3-Axis Metrics)')
print('='*60)

results_data = []
for model_name, result_file in results_by_method.items():
    result_path = OUTPUT_DIR / result_file
    if result_path.exists():
        with open(result_path, 'r') as f:
            results = json.load(f)
        
        # Extract the 3 PULSE metrics
        d_unlearn = results.get('D_unlearn_acc', 'N/A')
        d_retain = results.get('D_retain_acc', 'N/A')
        mmbench = results.get('MMBench', 'N/A')
        
        results_data.append({
            'Method': model_name,
            'D_unlearn': d_unlearn,
            'D_retain': d_retain,
            'MMBench': mmbench,
        })

if results_data:
    mllmu_df = pd.DataFrame(results_data)
    print('\n' + mllmu_df.to_string(index=False))
    
    # Save MLLMU-Bench results
    csv_path = OUTPUT_DIR / 'mllmu_results.csv'
    mllmu_df.to_csv(csv_path, index=False)
    print(f'\n✅ MLLMU-Bench results saved to {csv_path}')
else:
    print('⚠️  No results available yet')

# ─── Prepare for cross-benchmark analysis ──────────────────────────────────
print('\n' + '='*60)
print('Next: Kendall\'s τ Analysis (requires FIUBench results)')
print('='*60)
print('\n📋 To compute Kendall\'s τ correlation:')
print('  1. Load FIUBench method rankings (from reproduction results)')
print('  2. Extract MLLMU-Bench method rankings from above table')
print('  3. Compute Kendall\'s τ between the two ranking orders')
print('  4. Compare CMLD values (multimodal vs text-only leakage)')
print('  5. Generate radar charts (3-axis: D_unlearn, D_retain, MMBench)')
print(f'\n📁 All results: {OUTPUT_DIR}/')

## Cell 6: Generate Results Summary

In [ ]:
print('\n' + '='*60)
print('Extension 2: MLLMU-Bench Generalization Study')
print('='*60)

print(f'\n📋 Proposal-Aligned Workflow (Option B: MLLMU-Bench Baselines):')
print(f'  ✅ 1. Fine-tuned {BASE_MODEL} with LoRA on MLLMU-Bench')
print(f'  ✅ 2. Trained GA, GD, KL methods (MLLMU-Bench baseline equivalents)')
print(f'  ✅ 3. Evaluated with 3 PULSE metrics: D_unlearn, D_retain, MMBench')
print(f'  ⏳ 4. Pending: Kendall\'s τ correlation analysis (FIUBench vs MLLMU-Bench)')
print(f'  ⏳ 5. Pending: CMLD cross-benchmark comparison')
print(f'  ⏳ 6. Pending: Radar chart visualizations (3-axis)')

print(f'\nImplementation Details:')
print(f'  • Model: {BASE_MODEL}')
print(f'  • Fine-tuning: LoRA (batch=4, lr=2e-5, epochs=5, data=ft_Data)')
print(f'  • Unlearning: LoRA (batch=4, lr=2e-5, epochs=1)')
print(f'  • Methods: GA, GD (GA_Difference), KL (KL_Min)')
print(f'  • Dataset: MLLMU-Bench ({FORGET_SPLIT} + {RETAIN_SPLIT})')
print(f'  • Data source: Pre-made splits in MLLMU-Bench dataset')

print(f'\nData Configuration:')
print(f'  Forget split: {FORGET_SPLIT} (unlearning target)')
print(f'  Retain split: {RETAIN_SPLIT} (retention verification)')
print(f'  Fine-tune data: ft_Data (500 rows)')
print(f'  Eval test set: Test_Set (500 rows)')

print(f'\nPrimary Analysis (Proposal Core):')
print(f'  🎯 Kendall\'s τ: FIUBench method rankings vs MLLMU-Bench rankings')
print(f'  🎯 CMLD comparison: Text-only leakage consistency across datasets')
print(f'  🎯 Radar charts: 3-axis (D_unlearn, D_retain, MMBench) per method')

print(f'\nKey Findings Target:')
print(f'  τ ≈ 1.0 → Method rankings generalize (best method stays best)')
print(f'  τ ≈ 0.0 → Benchmark choice matters (rankings change)')
print(f'  CMLD stable → Multimodal leakage consistent across datasets')

print(f'\nOutput Structure:')
print(f'  📁 Checkpoints: {CKPT_DIR}/')
print(f'     ├── vanilla_model/')
print(f'     ├── GA/')
print(f'     ├── GD/')
print(f'     └── KL/')
print(f'  📊 Results: {OUTPUT_DIR}/')
print(f'     ├── mllmu_results.csv (3 methods × 3 metrics)')
print(f'     ├── *_results.json (detailed per method)')
print(f'     └── *_eval.log (evaluation logs)')

print(f'\n✅ Extension 2 updated to use pre-made {FORGET_SPLIT}/{RETAIN_SPLIT} splits')